# TCR Repertoire Aging Analysis

Reproduces the key findings from [Britanova et al. 2016](https://doi.org/10.1084/jem.20151413):

* Pairwise TRB repertoire overlap between 41 donors spanning ages 6–90 years
* Overlap computed with **exact**, **Hamming:1**, and **Levenshtein:1** matching
* F-metric and D-metric vs age scatter plots
* Pool-sample overlap (each donor vs the pooled cohort)
* UMAP embedding from pairwise dissimilarities, coloured by age

Reference figure: Fig. 3 of Britanova et al. 2016 (repertoire divergence with age).

In [ ]:
# Environment versions
import platform, importlib
print('Python:', platform.python_version())
for pkg in ('numpy', 'pandas', 'scipy', 'matplotlib', 'seaborn', 'umap', 'mir'):
    try:
        m = importlib.import_module(pkg if pkg != 'umap' else 'umap')
        ver = getattr(m, '__version__', '?')
        print(f'  {pkg}: {ver}')
    except ImportError:
        print(f'  {pkg}: NOT INSTALLED')

In [ ]:
# Imports
from __future__ import annotations

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats

from mir.common.parser import VDJtoolsParser
from mir.common.repertoire import LocusRepertoire
from mir.common.pool import pool_samples
from mir.common.repertoire_dataset import RepertoireDataset
from mir.comparative.overlap import pairwise_overlap, pairwise_overlap_matrix

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='ticks', context='paper', font_scale=1.2)
rng = np.random.default_rng(42)

In [ ]:
# Paths — prefer the full benchmark dataset, fall back to test assets
REPO_ROOT = Path('.').resolve().parent
_BENCH = REPO_ROOT / 'airr_benchmark' / 'vdjtools_lite'
_ASSETS = REPO_ROOT / 'tests' / 'assets' / 'real_repertoires'
DATA_DIR = _BENCH if _BENCH.is_dir() else _ASSETS
META_PATH = DATA_DIR / 'metadata_aging.txt'
print(f'Data directory : {DATA_DIR}')
print(f'Metadata file  : {META_PATH}')

## 1 · Load metadata and repertoires

In [ ]:
# Load metadata
meta = pd.read_csv(META_PATH, sep='\t')
print(f'Metadata rows: {len(meta)}')
meta.head(5)

In [ ]:
# Load repertoires
parser = VDJtoolsParser(sep='\t')

reps: list[LocusRepertoire] = []
valid_meta_rows = []

t0 = time.perf_counter()
for _, row in meta.iterrows():
    p = DATA_DIR / row['file_name']
    if not p.exists():
        # also try test assets
        p = _ASSETS / row['file_name']
    if not p.exists():
        continue
    clones = parser.parse(str(p))
    reps.append(LocusRepertoire(clonotypes=clones, locus='TRB'))
    valid_meta_rows.append(row)

meta_valid = pd.DataFrame(valid_meta_rows).reset_index(drop=True)
sample_ids = list(meta_valid['sample_id'])
ages = list(meta_valid['age'].astype(int))

print(f'Loaded {len(reps)} repertoires in {time.perf_counter()-t0:.1f}s')
print(f'Age range: {min(ages)}–{max(ages)} years')
sizes = [r.clonotype_count for r in reps]
print(f'Clonotype counts: min={min(sizes):,}  median={int(np.median(sizes)):,}  max={max(sizes):,}')

## 2 · Pairwise overlap matrices

In [ ]:
# Number of parallel workers — use all physical cores
N_JOBS = -1

# Compute pairwise overlap for all three matching modes
results: dict[str, pd.DataFrame] = {}

for mode_label, metric, threshold in [
    ('exact', 'exact', 0),
    ('hamming:1', 'hamming', 1),
    ('levenshtein:1', 'levenshtein', 1),
]:
    t0 = time.perf_counter()
    df = pairwise_overlap_matrix(
        reps, sample_ids=sample_ids,
        metric=metric, threshold=threshold,
        n_jobs=N_JOBS,
    )
    elapsed = time.perf_counter() - t0
    n_pairs = len(df)
    results[mode_label] = df
    print(f'{mode_label:<16} {n_pairs:>5} pairs  {elapsed:.1f}s')

print('Done.')

In [ ]:
# Preview the exact-matching result table
results['exact'].head(5)

## 3 · Pool sample and per-sample overlap with pool

In [ ]:
# Build a pooled repertoire from all samples (unique junction_aa + V + J key)
from mir.common.pool import pool_samples

# pool_samples expects a RepertoireDataset or list; use direct pooling of LocusRepertoires
# by concatenating clonotypes (pool_samples keys on 'aavj' by default)
all_clones = []
for rep in reps:
    all_clones.extend(rep.clonotypes)

pool_rep = LocusRepertoire(clonotypes=all_clones, locus='TRB')
print(f'Pooled repertoire: {pool_rep.clonotype_count:,} clonotypes (sum of all samples)')

In [ ]:
# Compute overlap of each individual sample with the pool, for all modes
pool_results: dict[str, list[dict]] = {}

for mode_label, metric, threshold in [
    ('exact', 'exact', 0),
    ('hamming:1', 'hamming', 1),
    ('levenshtein:1', 'levenshtein', 1),
]:
    rows = []
    t0 = time.perf_counter()
    for sid, rep, age in zip(sample_ids, reps, ages):
        r = pairwise_overlap(rep, pool_rep, metric=metric, threshold=threshold)
        d = r.as_dict()
        d['sample_id'] = sid
        d['age'] = age
        rows.append(d)
    pool_results[mode_label] = pd.DataFrame(rows)
    print(f'{mode_label:<16} pool overlap done in {time.perf_counter()-t0:.1f}s')

print('Done.')

## 4 · F-metric and D-metric vs age

In [ ]:
def _make_age_df(df_pairs: pd.DataFrame, meta_df: pd.DataFrame) -> pd.DataFrame:
    """Add age1 and age2 columns from sample_id metadata; compute mean age per pair."""
    age_map = dict(zip(meta_df['sample_id'], meta_df['age']))
    df = df_pairs.copy()
    df['age1'] = df['sample_id_1'].map(age_map)
    df['age2'] = df['sample_id_2'].map(age_map)
    df['mean_age'] = (df['age1'] + df['age2']) / 2
    return df


# Convert to dissimilarity (higher = more different)
def _dissim_d(df: pd.DataFrame) -> pd.Series:
    max_d = df['d_metric'].max()
    return max_d - df['d_metric']

def _dissim_f(df: pd.DataFrame) -> pd.Series:
    return 1.0 - df['f_metric']

In [ ]:
# Attach age info to exact-matching pairwise table
df_exact_age = _make_age_df(results['exact'], meta_valid)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (metric_col, label) in zip(axes, [
    ('f_metric', 'F-metric'),
    ('d_metric', 'D-metric'),
]):
    sc = ax.scatter(
        df_exact_age['mean_age'],
        df_exact_age[metric_col],
        c=df_exact_age['mean_age'],
        cmap='plasma',
        s=18, alpha=0.6, linewidths=0,
    )
    # Regression line
    x = df_exact_age['mean_age'].values
    y = df_exact_age[metric_col].values
    mask = np.isfinite(x) & np.isfinite(y)
    slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
    xr = np.linspace(x[mask].min(), x[mask].max(), 100)
    ax.plot(xr, slope * xr + intercept, color='#d62728', lw=1.5)
    ax.set_xlabel('Mean pair age (years)')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs mean age (exact)\nr={r:.3f}, p={p:.2e}')
    plt.colorbar(sc, ax=ax, label='Mean age')

sns.despine()
fig.tight_layout()
plt.savefig('../notebooks/assets/aging_fd_vs_age.pdf', bbox_inches='tight', dpi=150)
plt.show()
print(f'N pairs: {len(df_exact_age):,}')

In [ ]:
# Pool-overlap: F and D vs individual age
df_pool_exact = pool_results['exact']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (metric_col, label) in zip(axes, [
    ('f_metric', 'F-metric (vs pool)'),
    ('d_metric', 'D-metric (vs pool)'),
]):
    sc = ax.scatter(
        df_pool_exact['age'],
        df_pool_exact[metric_col],
        c=df_pool_exact['age'],
        cmap='plasma',
        s=50, alpha=0.85, linewidths=0,
    )
    x = df_pool_exact['age'].values.astype(float)
    y = df_pool_exact[metric_col].values
    mask = np.isfinite(x) & np.isfinite(y)
    slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
    xr = np.linspace(x[mask].min(), x[mask].max(), 100)
    ax.plot(xr, slope * xr + intercept, color='#d62728', lw=1.5)
    ax.set_xlabel('Age (years)')
    ax.set_ylabel(label)
    ax.set_title(f'{label}\nr={r:.3f}, p={p:.2e}')
    plt.colorbar(sc, ax=ax, label='Age')

sns.despine()
fig.tight_layout()
plt.savefig('../notebooks/assets/aging_pool_vs_age.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 5 · UMAP embedding from pairwise dissimilarities

In [ ]:
def _to_symmetric_matrix(df_pairs: pd.DataFrame, metric_col: str, ids: list[str]) -> np.ndarray:
    """Convert long-format pairwise DataFrame to symmetric NxN matrix."""
    n = len(ids)
    id_to_idx = {sid: i for i, sid in enumerate(ids)}
    mat = np.zeros((n, n))
    for _, row in df_pairs.iterrows():
        i = id_to_idx[row['sample_id_1']]
        j = id_to_idx[row['sample_id_2']]
        v = row[metric_col]
        mat[i, j] = v
        mat[j, i] = v  # symmetric
    return mat


def _dissimilarity_matrix(sim_mat: np.ndarray, mode: str = 'f') -> np.ndarray:
    """Convert similarity matrix to dissimilarity: 1-F or max(D)-D."""
    if mode == 'f':
        return 1.0 - sim_mat
    else:  # d
        max_d = sim_mat.max()
        return max_d - sim_mat

In [ ]:
try:
    from umap import UMAP
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print('umap-learn not installed — skipping UMAP. Install with: pip install umap-learn')

In [ ]:
if HAS_UMAP:
    # Build dissimilarity matrix from F-metric (exact matching)
    f_sim = _to_symmetric_matrix(results['exact'], 'f_metric', sample_ids)
    f_dissim = _dissimilarity_matrix(f_sim, mode='f')
    np.fill_diagonal(f_dissim, 0.0)  # self-distance = 0

    # UMAP with precomputed distance matrix
    umap_model = UMAP(
        n_components=2,
        metric='precomputed',
        n_neighbors=min(10, len(reps) - 1),
        min_dist=0.1,
        random_state=42,
    )
    embedding = umap_model.fit_transform(f_dissim)
    print(f'UMAP embedding shape: {embedding.shape}')

In [ ]:
if HAS_UMAP:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # UMAP coloured by age
    ax = axes[0]
    sc = ax.scatter(
        embedding[:, 0], embedding[:, 1],
        c=ages, cmap='plasma', s=60, alpha=0.9, linewidths=0,
    )
    plt.colorbar(sc, ax=ax, label='Age (years)')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title('UMAP (1−F dissimilarity, exact matching)')

    # Also try hamming:1
    f_sim_h1 = _to_symmetric_matrix(results['hamming:1'], 'f_metric', sample_ids)
    f_dissim_h1 = _dissimilarity_matrix(f_sim_h1, mode='f')
    np.fill_diagonal(f_dissim_h1, 0.0)

    umap_h1 = UMAP(
        n_components=2, metric='precomputed',
        n_neighbors=min(10, len(reps) - 1),
        min_dist=0.1, random_state=42,
    )
    emb_h1 = umap_h1.fit_transform(f_dissim_h1)

    ax = axes[1]
    sc = ax.scatter(
        emb_h1[:, 0], emb_h1[:, 1],
        c=ages, cmap='plasma', s=60, alpha=0.9, linewidths=0,
    )
    plt.colorbar(sc, ax=ax, label='Age (years)')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title('UMAP (1−F dissimilarity, Hamming:1)')

    sns.despine()
    fig.tight_layout()
    plt.savefig('../notebooks/assets/aging_umap.pdf', bbox_inches='tight', dpi=150)
    plt.show()

## 6 · Compare exact vs Hamming:1 vs Levenshtein:1

In [ ]:
# Overlay F-metric distribution by mode
fig, ax = plt.subplots(figsize=(8, 4))
colours = {'exact': '#1f77b4', 'hamming:1': '#ff7f0e', 'levenshtein:1': '#2ca02c'}

for mode_label, df in results.items():
    f_vals = df['f_metric'].dropna()
    ax.hist(f_vals, bins=40, alpha=0.55, label=mode_label,
            color=colours[mode_label], density=True)

ax.set_xlabel('F-metric')
ax.set_ylabel('Density')
ax.set_title('F-metric distribution by matching mode')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics per mode
summary_rows = []
for mode_label, df in results.items():
    for metric_col in ('f_metric', 'd_metric', 'jaccard', 'morisita_horn'):
        vals = df[metric_col].dropna()
        summary_rows.append({
            'mode': mode_label, 'metric': metric_col,
            'mean': vals.mean(), 'median': vals.median(),
            'std': vals.std(), 'max': vals.max(),
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))

## 7 · Age correlation summary

In [ ]:
# Spearman correlation of metric vs age for pool-overlap results
print(f'\nPool-overlap: Spearman correlation with age')
print(f'{"Mode":<18} {"Metric":<15} {"rho":>8} {"p-value":>12}')
print('-' * 56)

for mode_label, df in pool_results.items():
    for metric_col in ('f_metric', 'd_metric'):
        vals = df[metric_col].values
        age_vals = df['age'].values
        mask = np.isfinite(vals) & np.isfinite(age_vals)
        rho, p = stats.spearmanr(age_vals[mask], vals[mask])
        print(f'{mode_label:<18} {metric_col:<15} {rho:>8.3f} {p:>12.3e}')